# StockSense Phase 1: Data Understanding, Quality Audit & Master Integration

## Project Context & Objectives
**StockSense** is an end-to-end AI-driven inventory intelligence and demand forecasting system built for retail operations.

In **Phase 1**, our core objectives are:
1. **Raw Data Ingestion & Profiling:** Ingest and profile all five raw data sources without modifying raw files.
2. **Comprehensive Data Quality Audit:** Explicitly detect and handle real-world retail anomalies (orphan foreign keys, category casing inconsistencies, negative quantities/prices, invalid discount/hour values, missing external variables).
3. **Inventory Arithmetic Reconciliation:** Formally validate warehouse stock movements:
   $$\text{Expected Closing} = \text{Opening} + \text{Received} - \text{Sold}$$
   $$\text{Inventory Balance Gap} = \text{Closing} - \text{Expected Closing}$$
   Documenting gaps with flags rather than blindly overwriting data.
4. **POS vs. WMS Demand Discrepancy Analysis:** Compare store-level transaction sales against warehouse inventory depletion (`inventory.sold`), preserving both operational signals.
5. **Sparse History & Intermittent Demand Detection:** Profile the trading activity of all 1,200 Store $\times$ Product combinations over the 31-day window.
6. **Master Analytical Dataset Construction:** Produce the strictly required single consolidated table at:
   > **`ONE ROW = ONE DATE × ONE STORE × ONE PRODUCT`**
7. **Data Leakage Classification & Automated Validation Suite:** Enforce strict feature boundary rules and execute 15 automated validation checks.

In [1]:
import warnings, hashlib
from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 1000)

# Determine Project Root whether executed from repo root or notebooks/ directory
CWD = Path.cwd().resolve()
ROOT = CWD if (CWD / 'data').exists() else CWD.parent

RAW_DIR    = ROOT / 'data' / 'raw'
CLEAN_DIR  = ROOT / 'data' / 'cleaned'
PROC_DIR   = ROOT / 'data' / 'processed'
REPORT_DIR = ROOT / 'reports'

for d in [RAW_DIR, CLEAN_DIR, PROC_DIR, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Central Data Quality & Audit Trail Logs
AUDIT, DQ_LOG = [], []

def log_dq(ds, col, issue, cnt, pct, action, just):
    DQ_LOG.append(dict(dataset=ds, column=col, issue=issue, count=int(cnt),
                       percentage=round(float(pct), 4), action=action, justification=just))

def log_audit(ds, rid, fld, old, new, rsn):
    AUDIT.append(dict(dataset=ds, row_id=str(rid), field=fld,
                      old_value=str(old), new_value=str(new), reason=rsn))

def md5(p):
    return hashlib.md5(Path(p).read_bytes()).hexdigest()

print(f"Project Root: {ROOT}")
print(f"Data directories initialized successfully.")

Project Root: D:\DS\DS Hackathon\DAY 2
Data directories initialized successfully.


## Step 1: Raw Data Ingestion & Schema Profiling
We load the five raw input CSV files:
- `transactions.csv`: Point-of-Sale (POS) basket transactions.
- `products.csv`: Product catalog, MRP, cost price, shelf life, and supplier.
- `stores.csv`: Store physical metadata, format type, area, and city.
- `inventory.csv`: Warehouse management system (WMS) daily stock balances.
- `external_factors.csv`: City-level weather, holidays, and promotional events.

In [2]:
raw_tx  = pd.read_csv(RAW_DIR / 'transactions.csv')
raw_pr  = pd.read_csv(RAW_DIR / 'products.csv')
raw_st  = pd.read_csv(RAW_DIR / 'stores.csv')
raw_inv = pd.read_csv(RAW_DIR / 'inventory.csv')
raw_ext = pd.read_csv(RAW_DIR / 'external_factors.csv')

SCHEMA = {}
profile_rows = []
for name, df in [('transactions', raw_tx), ('products', raw_pr),
                  ('stores', raw_st), ('inventory', raw_inv),
                  ('external_factors', raw_ext)]:
    nulls = int(df.isnull().sum().sum())
    dups  = int(df.duplicated().sum())
    mem_mb = df.memory_usage(deep=True).sum() / (1024 * 1024)
    SCHEMA[name] = dict(rows=len(df), columns=len(df.columns),
                        column_names=list(df.columns), dtypes=df.dtypes.astype(str).to_dict(),
                        null_counts=df.isnull().sum().to_dict(), duplicate_rows=dups,
                        memory_bytes=int(df.memory_usage(deep=True).sum()))
    profile_rows.append({'Dataset': name, 'Rows': f"{len(df):,}", 'Cols': len(df.columns), 
                         'Duplicates': dups, 'Null Cells': nulls, 'Memory (MB)': f"{mem_mb:.2f}"})

df_profile = pd.DataFrame(profile_rows)
display(df_profile)

,Dataset,Rows,Cols,Duplicates,Null Cells,Memory (MB)
0,transactions,"151,928",11,25,9,60.35
1,products,120,8,0,0,0.04
2,stores,10,6,0,0,0.00
3,inventory,"26,907",9,0,2,6.06
4,external_factors,310,8,0,7,0.05


## Step 2: Reference Master Set Validation
Before auditing child records in transactions and inventory, we extract the authoritative set of store and product IDs from the dimension tables. Any IDs appearing downstream that are absent here represent **orphan foreign keys**.

In [3]:
VALID_STORE_IDS   = set(raw_st['store_id'])
VALID_PRODUCT_IDS = set(raw_pr['product_id'])

print(f"Authoritative Stores Count: {len(VALID_STORE_IDS)} -> {sorted(VALID_STORE_IDS)}")
print(f"Authoritative Products Count: {len(VALID_PRODUCT_IDS)} SKUs (e.g., {sorted(list(VALID_PRODUCT_IDS))[:5]}...)")

Authoritative Stores Count: 10 -> ['S01', 'S02', 'S03', 'S04', 'S05', 'S06', 'S07', 'S08', 'S09', 'S10']
Authoritative Products Count: 120 SKUs (e.g., ['P101', 'P102', 'P103', 'P104', 'P105']...)


## Step 3: Cleaning Products Master
### Identified Issues:
1. **Category Casing & Whitespace Variations:** Categories contain inconsistent casing and trailing whitespace (`beverage`, `BEVERAGES`, `DAIRY`, `SNACKS`, `household`, `Staples `).
2. **Economic Viability Check:** Verify whether `cost_price > mrp` occurs anywhere.
3. **Supplier Integrity:** Detect dummy/placeholder supplier IDs (`SUP999`).

In [4]:
pr = raw_pr.copy()
CATEGORY_MAP = {
    'beverage': 'Beverages', 'BEVERAGES': 'Beverages',
    'DAIRY': 'Dairy', 'SNACKS': 'Snacks',
    'household': 'Household', 'Staples ': 'Staples'
}

before_cats = pr['category'].value_counts().to_dict()
pr['category'] = pr['category'].str.strip().replace(CATEGORY_MAP)
n_std = sum(v for k, v in before_cats.items() if k in CATEGORY_MAP or k != k.strip())

log_dq('products', 'category', 'Case/spelling inconsistency', n_std, n_std / len(pr) * 100,
       'Standardised to canonical Title-case name',
       '6 variants (beverage, BEVERAGES, DAIRY, SNACKS, household, Staples[space]) mapped.')

for old, new in CATEGORY_MAP.items():
    if before_cats.get(old, 0) > 0:
        log_audit('products', f'cat={old}', 'category', old, new, f'Standardised {old!r}')

cost_gt_mrp = (pr['cost_price'] > pr['mrp']).sum()
log_dq('products', 'cost_price/mrp', 'cost_price > mrp', cost_gt_mrp, cost_gt_mrp / len(pr) * 100,
       'No action', 'PASS -- no violations')

sup_anm = (pr['supplier_id'] == 'SUP999').sum()
pr['invalid_supplier_flag'] = (pr['supplier_id'] == 'SUP999').astype(int)
if sup_anm:
    log_dq('products', 'supplier_id', 'Invalid supplier ID (SUP999)', sup_anm, sup_anm / len(pr) * 100,
           'Flagged; retained', 'SUP999 absent from supplier master.')

pr.to_csv(CLEAN_DIR / 'products_clean.csv', index=False)
print(f"Products Cleaned & Saved. Unique Categories: {sorted(pr['category'].unique())}")
print(f"Supplier SUP999 flagged: {sup_anm} rows | Cost > MRP violations: {cost_gt_mrp}")

Products Cleaned & Saved. Unique Categories: ['Bakery', 'Beverages', 'Dairy', 'Frozen', 'Fruits & Vegetables', 'Household', 'Personal Care', 'Snacks', 'Staples']
Supplier SUP999 flagged: 1 rows | Cost > MRP violations: 0


## Step 4: Cleaning Stores Master
Verify that the store dimension has complete attributes (store type, city, floor area, average daily footfall) with zero missingness and zero duplicates.

In [5]:
st = raw_st.copy()
log_dq('stores', 'ALL', 'Missing values', 0, 0.0, 'No action', 'PASS')
log_dq('stores', 'ALL', 'Duplicate rows',  0, 0.0, 'No action', 'PASS')

st.to_csv(CLEAN_DIR / 'stores_clean.csv', index=False)
print("Stores Master validated: 10 stores, 0 nulls, 0 duplicate rows. Saved to stores_clean.csv")
display(st.head(10))

Stores Master validated: 10 stores, 0 nulls, 0 duplicate rows. Saved to stores_clean.csv


,store_id,city,store_type,floor_area_sqft,avg_daily_customers,region
0,S01,Coimbatore,Supermarket,8500,1250,West
1,S02,Chennai,Hypermarket,18000,2850,North
2,S03,Madurai,Express,4200,720,South
3,S04,Salem,Supermarket,7600,1080,Central
4,S05,Tiruchirappalli,Hypermarket,14500,2100,Central
5,S06,Erode,Supermarket,6800,980,West
6,S07,Tiruppur,Express,5000,850,West
7,S08,Tirunelveli,Supermarket,7900,1120,South
8,S09,Vellore,Hypermarket,13200,1950,North
9,S10,Thanjavur,Supermarket,7200,1010,East


## Step 5: Cleaning Transactions (POS Traps & Anomaly Handling)
In high-volume POS feeds, several critical anomalies must be resolved:
1. **Date Windowing:** Flag fringe dates (July 31 and Sept 01) outside the main August 1-31 analysis window.
2. **Exact Duplicate Transactions:** 25 exact duplicate rows are removed (keeping first occurrence).
3. **Duplicate Transaction IDs:** Duplicate transaction IDs are flagged (`dup_txid_flag=1`) but retained as they represent POS receipt sequence collisions.
4. **Orphan Keys:** Detect store `S99` and product `P9999` not present in reference sets.
5. **Physical Impossible Values:**
   - Non-positive quantities (`quantity <= 0`): Flagged (`invalid_qty_flag=1`).
   - Non-positive prices (`selling_price <= 0`): Flagged (`invalid_price_flag=1`) and excluded from revenue aggregation.
   - Out-of-range discounts ($<0$ or $>100$): Flagged and set to `NaN`.
   - Out-of-range hours ($<0$ or $>23$): Flagged and set to `NaN`.
   - Missing customer IDs, payment modes, and hours: Preserved as `NaN`.

In [6]:
tx = raw_tx.copy()
tx_orig_len = len(tx)
tx['_idx'] = tx.index
tx['date'] = pd.to_datetime(tx['date'], errors='coerce')

# Fringe dates
date_lo, date_hi = pd.Timestamp('2026-08-01'), pd.Timestamp('2026-08-31')
tx['outside_main_window_flag'] = ((tx['date'] < date_lo) | (tx['date'] > date_hi)).astype(int)
n_out = tx['outside_main_window_flag'].sum()
log_dq('transactions', 'date', 'Outside Aug 01-31 window', n_out, n_out / len(tx) * 100,
       'Flagged; retained', '2 fringe records (Jul-31, Sep-01).')

# Exact duplicates
exact_dup = tx.duplicated(subset=tx.columns.difference(['_idx']), keep='first')
n_dup = exact_dup.sum()
log_dq('transactions', 'ALL', 'Exact duplicate rows', n_dup, n_dup / len(tx) * 100,
       'Removed -- kept first occurrence', 'Identical on all fields incl. transaction_id.')
for ri in tx[exact_dup]['_idx'].tolist():
    log_audit('transactions', ri, 'ALL', 'DUPLICATE', 'REMOVED', 'Exact duplicate row')
tx = tx[~exact_dup].copy()

# Duplicate transaction IDs
dup_id = tx.duplicated(subset=['transaction_id'], keep=False)
n_dup_id = dup_id.sum()
log_dq('transactions', 'transaction_id', 'Dup tx_id post-dedup', n_dup_id, n_dup_id / len(tx) * 100,
       'Flagged dup_txid_flag=1; retained', 'ID-generation errors.')
tx['dup_txid_flag'] = dup_id.astype(int)

# Orphan Store & Product IDs
tx['orphan_store_flag']   = (~tx['store_id'].isin(VALID_STORE_IDS)).astype(int)
tx['orphan_product_flag'] = (~tx['product_id'].isin(VALID_PRODUCT_IDS)).astype(int)
n_os = tx['orphan_store_flag'].sum()
n_op = tx['orphan_product_flag'].sum()
log_dq('transactions', 'store_id', 'Orphan store_id (S99)', n_os, n_os / len(tx) * 100,
       'Flagged orphan_store_flag=1; excluded from master', 'S99 not in stores reference.')
log_dq('transactions', 'product_id', 'Orphan product_id (P9999)', n_op, n_op / len(tx) * 100,
       'Flagged orphan_product_flag=1; excluded from master', 'P9999 not in products reference.')

# Negative/zero quantity & price
inv_qty = tx['quantity'] <= 0
n_iq = inv_qty.sum()
log_dq('transactions', 'quantity', 'Non-positive quantity', n_iq, n_iq / len(tx) * 100,
       'Flagged invalid_qty_flag=1; retained', 'Negative=possible return; zero=impossible.')
tx['invalid_qty_flag'] = inv_qty.astype(int)

inv_prc = tx['selling_price'] <= 0
n_ip = inv_prc.sum()
log_dq('transactions', 'selling_price', 'Non-positive selling price', n_ip, n_ip / len(tx) * 100,
       'Flagged invalid_price_flag=1; excluded from revenue', 'Negative price impossible.')
tx['invalid_price_flag'] = inv_prc.astype(int)

# Impossible discounts & hours
inv_disc = (tx['discount_pct'] < 0) | (tx['discount_pct'] > 100)
n_id = inv_disc.sum()
log_dq('transactions', 'discount_pct', 'Discount outside [0,100]', n_id, n_id / len(tx) * 100,
       'Flagged; value set to NaN', 'Values 110 and -15 impossible. Set NaN.')
tx['invalid_discount_flag'] = inv_disc.astype(int)
tx.loc[inv_disc, 'discount_pct'] = np.nan

inv_hr = (tx['hour'] < 0) | (tx['hour'] > 23)
n_ih = inv_hr.sum()
log_dq('transactions', 'hour', 'Hour outside [0,23]', n_ih, n_ih / len(tx) * 100,
       'Flagged; value set to NaN', 'Values 27 and -3 impossible clock values. Set NaN.')
tx['invalid_hour_flag'] = inv_hr.astype(int)
tx.loc[inv_hr, 'hour'] = np.nan

for col in ['customer_id', 'payment_mode', 'hour']:
    n_miss = tx[col].isna().sum()
    if n_miss:
        log_dq('transactions', col, f'Missing {col}', n_miss, n_miss / len(tx) * 100,
               'Retained as NaN', 'Preserved without synthetic imputation.')

tx['payment_mode'] = tx['payment_mode'].str.strip().str.title()
tx.drop(columns=['_idx'], inplace=True)
tx.to_csv(CLEAN_DIR / 'transactions_clean.csv', index=False)

print(f"Transactions Cleaned: {len(tx):,} rows retained ({tx_orig_len - len(tx)} exact duplicates removed).")
print(f"Anomalies: Orphan Store={n_os}, Orphan Product={n_op}, Invalid Qty={n_iq}, Invalid Price={n_ip}, Invalid Disc={n_id}, Invalid Hr={n_ih}")

Transactions Cleaned: 151,903 rows retained (25 exact duplicates removed).
Anomalies: Orphan Store=1, Orphan Product=1, Invalid Qty=3, Invalid Price=1, Invalid Disc=2, Invalid Hr=2


## Step 6: Cleaning Inventory & Arithmetic Stock Reconciliation
A foundational pillar of warehouse inventory accounting:
$$\text{expected\_closing} = \text{opening} + \text{received} - \text{sold}$$
$$\text{inventory\_balance\_gap} = \text{closing} - \text{expected\_closing}$$

### Reconciliation Classification:
- **`CONSISTENT`**: $\text{gap} == 0$.
- **`MINOR_MISMATCH`**: $0 < |\text{gap}| \le 5$ (operational rounding, scanning delays).
- **`MAJOR_MISMATCH`**: $|\text{gap}| > 5$ (shrinkage, unrecorded breakage, theft).
- **`INVALID_INPUT`**: Physically impossible opening stock or received quantities ($<0$).
- **`INSUFFICIENT_DATA`**: Missing closing or received stock entries.

> **Principle:** We **never** overwrite closing inventory to force agreement; doing so hides physical inventory leakage and inventory inaccuracies.

In [7]:
inv = raw_inv.copy()
inv['date'] = pd.to_datetime(inv['date'], errors='coerce')
inv.rename(columns={'store': 'store_id', 'product': 'product_id'}, inplace=True)

inv['orphan_store_flag']   = (~inv['store_id'].isin(VALID_STORE_IDS)).astype(int)
inv['orphan_product_flag'] = (~inv['product_id'].isin(VALID_PRODUCT_IDS)).astype(int)

inv_open_bad = inv['opening'] < 0
inv['invalid_inventory_flag'] = inv_open_bad.astype(int)
inv_recv_bad = inv['received'] < 0
inv.loc[inv_recv_bad, 'invalid_inventory_flag'] = 1

inv_lead_bad = inv['lead_days'] <= 0
inv['invalid_lead_flag'] = inv_lead_bad.astype(int)

# Arithmetic reconciliation
inv['expected_closing']      = inv['opening'] + inv['received'].fillna(0) - inv['sold']
inv['inventory_balance_gap'] = inv['closing'] - inv['expected_closing']

def rec_status(row):
    if pd.isna(row['closing']) or pd.isna(row['received']):
        return 'INSUFFICIENT_DATA'
    if row.get('invalid_inventory_flag', 0) == 1 or row.get('invalid_lead_flag', 0) == 1:
        return 'INVALID_INPUT'
    g = abs(row['inventory_balance_gap'])
    if g == 0:
        return 'CONSISTENT'
    if g <= 5:
        return 'MINOR_MISMATCH'
    return 'MAJOR_MISMATCH'

inv['reconciliation_status'] = inv.apply(rec_status, axis=1)
rc_counts = inv['reconciliation_status'].value_counts()

print("Inventory Arithmetic Reconciliation Summary:")
for status, count in rc_counts.items():
    print(f"  {status:<20}: {count:>6,}  ({count / len(inv) * 100:.2f}%)")

inv.to_csv(CLEAN_DIR / 'inventory_clean.csv', index=False)
print("Saved: inventory_clean.csv")

Inventory Arithmetic Reconciliation Summary:
  CONSISTENT          : 26,899  (99.97%)
  INVALID_INPUT       :      4  (0.01%)
  MAJOR_MISMATCH      :      2  (0.01%)
  INSUFFICIENT_DATA   :      2  (0.01%)
Saved: inventory_clean.csv


## Step 7: Cleaning External Factors (Weather & Calendar Events)
### Anomalies Addressed:
1. **City Spelling Typo:** The entry `'ChennaI'` (with a capital I) is corrected to `'Chennai'`.
2. **Missing Temperature & Rain:** Imputed using city-level monthly medians, accompanied by indicator flags (`temp_imputed_flag`, `rain_imputed_flag`).
3. **Date+City Deduplication:** Fixing `'ChennaI'` created two records for Chennai on 2026-08-25. We aggregate cleanly: `mean` for numerical variables (`temp_c`, `rain_mm`) and `max` for binary flags (`holiday`, `festival`, `weekend`, `local_event`).
4. **Calendar Date Coverage:** Flag dates where inventory transactions exist beyond external factors data.

In [8]:
ext = raw_ext.copy()
ext['date'] = pd.to_datetime(ext['date'], errors='coerce')

# Fix city typo
typo_mask = ext['city'] == 'ChennaI'
n_ct = typo_mask.sum()
log_dq('external_factors', 'city', "City typo: 'ChennaI'", n_ct, n_ct / len(ext) * 100,
       "Corrected to 'Chennai'", "Obvious typo (capital I).")
ext.loc[typo_mask, 'city'] = 'Chennai'

# Impute missing weather
ext['temp_imputed_flag'] = ext['temp_c'].isna().astype(int)
ext['temp_c'] = ext.groupby('city')['temp_c'].transform(lambda x: x.fillna(x.median()))

ext['rain_imputed_flag'] = ext['rain_mm'].isna().astype(int)
ext['rain_mm'] = ext.groupby('city')['rain_mm'].transform(lambda x: x.fillna(x.median()))

# Deduplicate date+city after typo correction
ext_dedup = ext.groupby(['date', 'city'], as_index=False).agg({
    'temp_c': 'mean', 'rain_mm': 'mean',
    'holiday': 'max', 'festival': 'max', 'weekend': 'max', 'local_event': 'max',
    'temp_imputed_flag': 'max', 'rain_imputed_flag': 'max'
})

ext = ext_dedup
ext.to_csv(CLEAN_DIR / 'external_factors_clean.csv', index=False)
print(f"External factors cleaned and deduplicated. Shape: {ext.shape}")
display(ext.head())

External factors cleaned and deduplicated. Shape: (309, 10)


,date,city,temp_c,rain_mm,holiday,festival,weekend,local_event,temp_imputed_flag,rain_imputed_flag
0,2026-08-01,Chennai,30.6,3.6,0,0,1,0,0,0
1,2026-08-01,Coimbatore,30.1,0.8,0,0,1,0,0,0
2,2026-08-01,Erode,31.2,0.2,0,0,1,0,0,0
3,2026-08-01,Madurai,31.9,0.0,0,0,1,0,0,0
4,2026-08-01,Salem,31.4,2.1,0,0,1,0,0,0


## Step 8: Transaction Demand vs. Inventory Sold Reconciliation (POS vs. WMS)
In enterprise retail, **cash register POS sales** (`transaction_demand`) and **warehouse inventory depletions** (`inventory.sold`) almost never match 1:1 due to:
- Shrinkage (theft, spoilage, breakage)
- Scanning delays & lag between order processing and physical shelf replenishment
- Returns and batch adjustments

We compute:
$$\text{inventory\_demand\_gap} = \text{transaction\_demand} - \text{inventory\_sold}$$

In [9]:
tx_clean  = pd.read_csv(CLEAN_DIR / 'transactions_clean.csv', parse_dates=['date'])
inv_clean = pd.read_csv(CLEAN_DIR / 'inventory_clean.csv', parse_dates=['date'])

tx_dem = (tx_clean.groupby(['date', 'store_id', 'product_id'])['quantity']
          .sum().reset_index().rename(columns={'quantity': 'transaction_demand'}))
inv_dem = inv_clean[['date', 'store_id', 'product_id', 'sold']].rename(columns={'sold': 'inventory_sold'})

rec_df = pd.merge(tx_dem, inv_dem, on=['date', 'store_id', 'product_id'], how='outer')
rec_df['inventory_demand_gap'] = rec_df['transaction_demand'] - rec_df['inventory_sold']

n_match   = rec_df.dropna(subset=['transaction_demand', 'inventory_sold']).shape[0]
n_tx_only = rec_df['inventory_sold'].isna().sum()
n_iv_only = rec_df['transaction_demand'].isna().sum()
n_perf    = (rec_df['inventory_demand_gap'] == 0).sum()

print(f"Matched Observations (both POS and WMS present): {n_match:,}")
print(f"POS Only Observations: {n_tx_only:,} | WMS Only Observations: {n_iv_only:,}")
print(f"Perfect Agreement (gap == 0): {n_perf:,} ({(n_perf / n_match) * 100:.2f}%)")
print(f"Gap Mean: {rec_df['inventory_demand_gap'].mean():.2f} | Std: {rec_df['inventory_demand_gap'].std():.2f}")

log_dq('transactions/inventory', 'quantity/sold', 'TX demand != inv sold',
       int((rec_df['inventory_demand_gap'] != 0).sum()),
       (rec_df['inventory_demand_gap'] != 0).sum() / len(rec_df) * 100,
       'Both retained; inventory_demand_gap created',
       'POS vs WMS discrepancies expected. Replacing one with the other is data fabrication.')

rec_df.to_csv(PROC_DIR / 'tx_inv_reconciliation.csv', index=False)
print("Saved: tx_inv_reconciliation.csv")

Matched Observations (both POS and WMS present): 21,010
POS Only Observations: 8,071 | WMS Only Observations: 5,897
Perfect Agreement (gap == 0): 59 (0.28%)
Gap Mean: -149.97 | Std: 102.11
Saved: tx_inv_reconciliation.csv


## Step 9: Sparse History Analysis (Intermittent Demand SKUs)
We construct the full Cartesian grid:
$$31 \text{ Dates} \times 10 \text{ Stores} \times 120 \text{ Products} = 37,200 \text{ possible daily observations}$$
Across all 1,200 Store $\times$ Product combinations, we compute:
- `zero_demand_ratio`: Proportion of days with 0 sales.
- `active_days`: Number of days with $>0$ sales.
- `sparse_history_flag`: Triggered if `zero_demand_ratio > 0.70` or `active_days < 7`.

In [10]:
all_dates    = pd.date_range('2026-08-01', '2026-08-31', freq='D')
all_stores   = sorted(VALID_STORE_IDS)
all_products = sorted(VALID_PRODUCT_IDS)

full_grid = pd.MultiIndex.from_product(
    [all_dates, all_stores, all_products],
    names=['date', 'store_id', 'product_id']
).to_frame(index=False)

tx_pos = (tx_clean[(tx_clean['orphan_store_flag'] == 0) & 
                   (tx_clean['orphan_product_flag'] == 0) & 
                   (tx_clean['quantity'] > 0)]
          .groupby(['date', 'store_id', 'product_id'])['quantity'].sum()
          .reset_index().rename(columns={'quantity': 'units_sold'}))

sg = pd.merge(full_grid, tx_pos, on=['date', 'store_id', 'product_id'], how='left')
sg['units_sold'] = sg['units_sold'].fillna(0)

sp_stats = (sg.groupby(['store_id', 'product_id'])
    .agg(history_length=('date', 'count'),
         zero_demand_days=('units_sold', lambda x: (x == 0).sum()),
         total_demand=('units_sold', 'sum'),
         active_days=('units_sold', lambda x: (x > 0).sum()))
    .reset_index())

sp_stats['zero_demand_ratio'] = sp_stats['zero_demand_days'] / sp_stats['history_length']
sp_stats['mean_daily_demand'] = sp_stats['total_demand'] / sp_stats['active_days'].replace(0, np.nan)
sp_stats['sparse_history_flag'] = ((sp_stats['zero_demand_ratio'] > 0.70) | (sp_stats['active_days'] < 7)).astype(int)

n_sparse = sp_stats['sparse_history_flag'].sum()
n_total  = len(sp_stats)
n_never  = (sp_stats['zero_demand_ratio'] == 1.0).sum()

print(f"Total Store x Product Combinations: {n_total:,}")
print(f"Sparse History Combinations (>70% zeros or <7 active days): {n_sparse:,} ({n_sparse / n_total * 100:.1f}%)")
print(f"Never-sold Combinations: {n_never}")

sp_stats.to_csv(PROC_DIR / 'sparse_history_analysis.csv', index=False)
print("Saved: sparse_history_analysis.csv")

Total Store x Product Combinations: 1,200
Sparse History Combinations (>70% zeros or <7 active days): 71 (5.9%)
Never-sold Combinations: 1
Saved: sparse_history_analysis.csv


## Step 10: Transaction Aggregation to Master Grain
We aggregate the transaction feed into the required grain:
> **`ONE ROW = ONE DATE × ONE STORE × ONE PRODUCT`**

Aggregations applied:
- `transaction_demand`: Sum of valid positive quantities.
- `revenue`: Sum of `quantity * selling_price` (positive values only).
- `transaction_count`: Total transaction events for this grain.
- `unique_customer_count`: Unique customer IDs.
- `avg_selling_price`: Weighted average realized price.
- `avg_discount_pct`: Mean valid discount percentage.
- `promotion_active`: `max(promotion_flag)` (1 if any transaction had a promotion active).
- `peak_hour`: Statistical mode of transaction hours.
- Anomaly counts: `invalid_qty_count`, `invalid_price_count`.

In [11]:
tx_agg = tx_clean[(tx_clean['orphan_store_flag'] == 0) & (tx_clean['orphan_product_flag'] == 0)].copy()
tx_vq  = tx_agg[tx_agg['invalid_qty_flag'] == 0]
tx_vqp = tx_agg[(tx_agg['invalid_qty_flag'] == 0) & (tx_agg['invalid_price_flag'] == 0)].copy()

KEY = ['date', 'store_id', 'product_id']

a_qty   = tx_vq.groupby(KEY)['quantity'].sum().reset_index().rename(columns={'quantity': 'transaction_demand'})
tx_vqp['revenue'] = tx_vqp['quantity'] * tx_vqp['selling_price']
a_rev   = tx_vqp.groupby(KEY)['revenue'].sum().reset_index()
a_cnt   = tx_agg.groupby(KEY)['transaction_id'].count().reset_index().rename(columns={'transaction_id': 'transaction_count'})
a_cust  = tx_agg.groupby(KEY)['customer_id'].nunique().reset_index().rename(columns={'customer_id': 'unique_customer_count'})
a_price = tx_vqp.groupby(KEY)['selling_price'].mean().reset_index().rename(columns={'selling_price': 'avg_selling_price'})
a_disc  = tx_agg[tx_agg['invalid_discount_flag'] == 0].groupby(KEY)['discount_pct'].mean().reset_index().rename(columns={'discount_pct': 'avg_discount_pct'})
a_promo = tx_agg.groupby(KEY)['promotion_flag'].max().reset_index().rename(columns={'promotion_flag': 'promotion_active'})
a_peak  = tx_agg.dropna(subset=['hour']).groupby(KEY)['hour'].agg(lambda x: x.mode().iloc[0] if len(x) > 0 else np.nan).reset_index().rename(columns={'hour': 'peak_hour'})
a_iqc   = tx_agg.groupby(KEY)['invalid_qty_flag'].sum().reset_index().rename(columns={'invalid_qty_flag': 'invalid_qty_count'})
a_ipc   = tx_agg.groupby(KEY)['invalid_price_flag'].sum().reset_index().rename(columns={'invalid_price_flag': 'invalid_price_count'})

daily_tx = a_qty.copy()
for agg_df in [a_rev, a_cnt, a_cust, a_price, a_disc, a_promo, a_peak, a_iqc, a_ipc]:
    daily_tx = pd.merge(daily_tx, agg_df, on=KEY, how='left')

daily_tx.to_csv(PROC_DIR / 'daily_transactions_agg.csv', index=False)
print(f"Daily Aggregated POS records: {len(daily_tx):,} rows")
display(daily_tx.head())

Daily Aggregated POS records: 29,079 rows


,date,store_id,product_id,transaction_demand,revenue,transaction_count,unique_customer_count,avg_selling_price,avg_discount_pct,promotion_active,peak_hour,invalid_qty_count,invalid_price_count
0,2026-07-31,S03,P133,2,360.0,1,1,180.000000,0.000000,0,13.0,0,0
1,2026-08-01,S01,P101,12,3360.0,6,6,280.000000,6.666667,1,14.0,0,0
2,2026-08-01,S01,P103,23,1308.0,14,14,56.357143,6.071429,1,13.0,0,0
3,2026-08-01,S01,P104,1,237.5,1,1,237.500000,5.000000,1,17.0,0,0
4,2026-08-01,S01,P105,4,152.0,1,1,38.000000,5.000000,1,12.0,0,0


## Step 11: Master Analytical Dataset Construction
We integrate the aggregated POS transactions with:
1. `products_clean.csv`: Static SKU attributes (category, brand, cost_price, mrp, shelf_life_days).
2. `stores_clean.csv`: Static store attributes (store_type, floor_area_sqft, city).
3. `inventory_clean.csv`: Point-in-time opening, received, closing, reorder level, lead days, and balance gap.
4. `external_factors_clean.csv`: Weather, temperature, holiday, festival, and weekend flags.
5. `sparse_history_analysis.csv`: Historical demand density flags.
6. `tx_inv_reconciliation.csv`: POS vs WMS demand gap.

In [12]:
prod_c = pd.read_csv(CLEAN_DIR / 'products_clean.csv')
stor_c = pd.read_csv(CLEAN_DIR / 'stores_clean.csv')
ext_c  = pd.read_csv(CLEAN_DIR / 'external_factors_clean.csv', parse_dates=['date'])
inv_c  = pd.read_csv(CLEAN_DIR / 'inventory_clean.csv',        parse_dates=['date'])

master = pd.merge(daily_tx, prod_c, on='product_id', how='left')
master = pd.merge(master, stor_c, on='store_id', how='left')

inv_join = inv_c[['date', 'store_id', 'product_id', 'opening', 'received', 'sold', 'closing',
                  'reorder_lvl', 'lead_days', 'inventory_balance_gap', 'reconciliation_status',
                  'invalid_inventory_flag', 'invalid_lead_flag', 'expected_closing',
                  'orphan_store_flag', 'orphan_product_flag']].rename(columns={
    'sold': 'inventory_sold',
    'orphan_store_flag': 'inv_orphan_store_flag',
    'orphan_product_flag': 'inv_orphan_product_flag'
})

master = pd.merge(master, inv_join, on=['date', 'store_id', 'product_id'], how='left')
ext_join = ext_c.rename(columns={'city': 'city_ext'})
master = pd.merge(master, ext_join, left_on=['date', 'city'], right_on=['date', 'city_ext'], how='left')
master.drop(columns=['city_ext'], inplace=True, errors='ignore')

master = pd.merge(master, sp_stats[['store_id', 'product_id', 'history_length', 'zero_demand_ratio',
                                     'active_days', 'sparse_history_flag', 'mean_daily_demand']],
                  on=['store_id', 'product_id'], how='left')

master = pd.merge(master, rec_df[['date', 'store_id', 'product_id', 'transaction_demand',
                                  'inventory_sold', 'inventory_demand_gap']],
                  on=['date', 'store_id', 'product_id'], how='left', suffixes=('', '_rec'))
master.drop(columns=[c for c in master.columns if c.endswith('_rec')], inplace=True, errors='ignore')

master['date'] = pd.to_datetime(master['date'])
dup_grains = master.duplicated(subset=['date', 'store_id', 'product_id']).sum()

print(f"Master Dataset Shape: {master.shape[0]:,} rows x {master.shape[1]} columns")
print(f"Grain Uniqueness Check (date x store_id x product_id duplicates): {dup_grains} -> {'PASS' if dup_grains == 0 else 'FAIL'}")

master.to_csv(PROC_DIR / 'master_dataset.csv', index=False)
print("Saved: master_dataset.csv")

Master Dataset Shape: 29,079 rows x 53 columns
Grain Uniqueness Check (date x store_id x product_id duplicates): 0 -> PASS
Saved: master_dataset.csv


## Step 12: Data Leakage Prevention Classification
To avoid lookahead bias and target leakage during predictive modeling, we classify all columns into strict behavioral categories:
- **`SAFE_HISTORICAL`**: Lagged demand, rolling aggregates, and point-in-time weather/calendar inputs.
- **`SAFE_STATIC_ATTRIBUTE`**: Time-invariant SKU and store attributes (category, brand, store_type, sqft).
- **`SAFE_DERIVED_DQ`**: Diagnostic quality indicators (gaps, reconciliation status, sparse history flags).
- **`DO_NOT_USE_AS_FEATURE`**: Contaminated target constructs (e.g. `expected_closing` contains contemporaneous sales information).

In [13]:
LEAKAGE = {
    'SAFE_HISTORICAL': [
        'transaction_demand', 'revenue', 'transaction_count', 'unique_customer_count',
        'avg_selling_price', 'avg_discount_pct', 'promotion_active', 'peak_hour',
        'opening', 'received', 'inventory_sold', 'closing', 'reorder_lvl', 'lead_days',
        'temp_c', 'rain_mm', 'holiday', 'festival', 'weekend', 'local_event'
    ],
    'SAFE_STATIC_ATTRIBUTE': [
        'category', 'sub_category', 'brand', 'mrp', 'cost_price', 'shelf_life_days',
        'supplier_id', 'city', 'store_type', 'floor_area_sqft', 'avg_daily_customers', 'region'
    ],
    'SAFE_DERIVED_DQ': [
        'inventory_balance_gap', 'reconciliation_status', 'inventory_demand_gap',
        'sparse_history_flag', 'zero_demand_ratio', 'history_length', 'active_days',
        'invalid_qty_count', 'invalid_price_count', 'invalid_inventory_flag', 'invalid_lead_flag',
        'temp_imputed_flag', 'rain_imputed_flag'
    ],
    'DO_NOT_USE_AS_FEATURE': [
        'expected_closing'
    ],
}

lk_path = REPORT_DIR / 'leakage_classification.md'
with open(lk_path, 'w', encoding='utf-8') as f:
    f.write('# Data Leakage Classification\n\nPhase 3 MUST NOT use DO_NOT_USE_AS_FEATURE columns as predictors.\n\n')
    for grp, cols in LEAKAGE.items():
        f.write(f'## {grp}\n')
        for c in cols:
            f.write(f'- {c}\n')
        f.write('\n')

print("Saved: reports/leakage_classification.md")

Saved: reports/leakage_classification.md


## Step 13: Automated Quality & Integrity Validation Suite
We execute 15 automated assertion checks against the pipeline outputs to ensure 100% compliance with challenge specifications.

In [14]:
mc = pd.read_csv(PROC_DIR / 'master_dataset.csv', parse_dates=['date'])
CHECKS = []

def chk(name, cond, detail=''):
    status = 'PASS' if cond else 'FAIL'
    CHECKS.append({'check': name, 'status': status, 'detail': detail})

chk('1. Master grain uniqueness (Date x Store x Product)', mc.duplicated(['date', 'store_id', 'product_id']).sum() == 0)
req = ['date', 'store_id', 'product_id', 'transaction_demand', 'revenue', 'avg_selling_price',
       'promotion_active', 'opening', 'inventory_sold', 'closing', 'reorder_lvl', 'lead_days',
       'inventory_balance_gap', 'reconciliation_status', 'inventory_demand_gap',
       'sparse_history_flag', 'category', 'brand', 'city', 'store_type']
miss = [c for c in req if c not in mc.columns]
chk('2. Core analytical columns present', len(miss) == 0, f'Missing: {miss}')
chk('3. Zero null dates', mc['date'].isna().sum() == 0)
chk('4. transaction_demand non-negative', (mc['transaction_demand'] < 0).sum() == 0)
chk('5. revenue non-negative', (mc['revenue'] < 0).sum() == 0)
chk('6. inventory_balance_gap present', 'inventory_balance_gap' in mc.columns)
valid_s = {'CONSISTENT', 'MINOR_MISMATCH', 'MAJOR_MISMATCH', 'INVALID_INPUT', 'INSUFFICIENT_DATA'}
actual_s = set(mc['reconciliation_status'].dropna().unique())
chk('7. reconciliation_status values valid', actual_s.issubset(valid_s), f'Unexpected: {actual_s - valid_s}')
sv = set(mc['sparse_history_flag'].dropna().unique())
chk('8. sparse_history_flag is strictly binary', sv.issubset({0, 1, 0.0, 1.0}))
raw_ok = all(md5(RAW_DIR / f) == md5(ROOT / 'dataset' / f) for f in
             ['transactions.csv', 'products.csv', 'stores.csv', 'inventory.csv', 'external_factors.csv'])
chk('9. Raw source files untouched (MD5 Checksum)', raw_ok)
cf = ['transactions_clean.csv', 'products_clean.csv', 'stores_clean.csv', 'inventory_clean.csv', 'external_factors_clean.csv']
chk('10. All cleaned dimension files exist', all((CLEAN_DIR / f).exists() for f in cf))
pf = ['master_dataset.csv', 'daily_transactions_agg.csv', 'tx_inv_reconciliation.csv', 'sparse_history_analysis.csv']
chk('11. All processed artifacts exist', all((PROC_DIR / f).exists() for f in pf))
future_cols = [c for c in mc.columns if 'next_' in c or 'future' in c]
chk('12. Zero future-derived lookahead columns', len(future_cols) == 0, f'Found: {future_cols}')
pc = pd.read_csv(CLEAN_DIR / 'products_clean.csv')
valid_cats = {'Beverages', 'Dairy', 'Snacks', 'Frozen', 'Bakery', 'Staples', 'Household', 'Personal Care', 'Fruits & Vegetables'}
bad_cats = set(pc['category']) - valid_cats
chk('13. Product categories standardized', len(bad_cats) == 0, f'Bad: {bad_cats}')
ec = pd.read_csv(CLEAN_DIR / 'external_factors_clean.csv')
chk("14. External city typo ('ChennaI') resolved", (ec['city'] == 'ChennaI').sum() == 0)
chk('15. Zero missing temperature values', ec['temp_c'].isna().sum() == 0)

df_checks = pd.DataFrame(CHECKS)
display(df_checks)

n_pass = sum(1 for c in CHECKS if c['status'] == 'PASS')
print(f"Validation Suite Result: {n_pass} / {len(CHECKS)} CHECKS PASSED.")

,check,status,detail
0,1. Master grain uniqueness (Date x Store x Pro...,PASS,
1,2. Core analytical columns present,PASS,Missing: []
2,3. Zero null dates,PASS,
3,4. transaction_demand non-negative,PASS,
4,5. revenue non-negative,PASS,
5,6. inventory_balance_gap present,PASS,
6,7. reconciliation_status values valid,PASS,Unexpected: set()
7,8. sparse_history_flag is strictly binary,PASS,
8,9. Raw source files untouched (MD5 Checksum),PASS,
9,10. All cleaned dimension files exist,PASS,


Validation Suite Result: 15 / 15 CHECKS PASSED.


## Step 14 & 15: Save Audit Trail & Generate Data Quality Report
We save the audit log of all record-level transformations and export the official `reports/data_quality_report.md`.

In [15]:
pd.DataFrame(AUDIT).to_csv(PROC_DIR / 'audit_trail.csv', index=False)
pd.DataFrame(CHECKS).to_csv(PROC_DIR / 'validation_results.csv', index=False)

dq_path = REPORT_DIR / 'data_quality_report.md'
now_s = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
inv_cr = pd.read_csv(CLEAN_DIR / 'inventory_clean.csv')
sp_cr  = pd.read_csv(PROC_DIR / 'sparse_history_analysis.csv')

with open(dq_path, 'w', encoding='utf-8') as f:
    f.write(f'# StockSense -- Data Quality Report\n**Phase 1** | {now_s}\n\n---\n\n')
    f.write('## 1. Dataset Overview\n\n| Dataset | Rows | Columns | Dup Rows | Missing |\n')
    f.write('|---------|------|---------|----------|---------|\n')
    for name, s in SCHEMA.items():
        f.write(f'| {name} | {s["rows"]:,} | {s["columns"]} | {s["duplicate_rows"]} | {sum(s["null_counts"].values())} |\n')
    f.write('\n## 2. Data Quality Issues\n\n| Dataset | Column | Issue | Count | % | Action | Justification |\n')
    f.write('|---------|--------|-------|-------|---|--------|---------------|\n')
    for row in DQ_LOG:
        f.write(f'| {row["dataset"]} | {row["column"]} | {row["issue"]} | {row["count"]} | {row["percentage"]:.2f}% | {row["action"]} | {row["justification"]} |\n')
    f.write('\n## 3. Inventory Reconciliation\n\n| Status | Count | % |\n|--------|-------|---|\n')
    for status, cnt in inv_cr['reconciliation_status'].value_counts().items():
        f.write(f'| {status} | {cnt:,} | {cnt / len(inv_cr) * 100:.2f}% |\n')
    f.write('\n## 4. TX vs Inventory Conclusion\n')
    f.write('> transaction_demand and inventory.sold are DIFFERENT signals (POS vs WMS). Both retained.\n')
    f.write('> inventory_demand_gap documents the discrepancy for Phase 3.\n\n')
    f.write(f'## 5. Sparse History\n- Total combos: {len(sp_cr):,}\n')
    f.write(f'- Sparse: {sp_cr["sparse_history_flag"].sum():,} ({sp_cr["sparse_history_flag"].mean() * 100:.1f}%)\n')
    f.write(f'- Never sold: {(sp_cr["zero_demand_ratio"] == 1.0).sum():,}\n\n')
    f.write('## 6. Validation Results\n\n| Check | Status | Detail |\n|-------|--------|--------|\n')
    for c in CHECKS:
        f.write(f'| {c["check"]} | {c["status"]} | {c.get("detail", "")} |\n')

print(f"Data Quality Report successfully compiled and saved to {dq_path}")
print("Phase 1 Pipeline Execution Complete.")

Data Quality Report successfully compiled and saved to D:\DS\DS Hackathon\DAY 2\reports\data_quality_report.md
Phase 1 Pipeline Execution Complete.
